In [ ]:
# NCrystal notebook settings
# title: ENDF export example
# menutitle: ENDF export
# shortkey: endfexport
# section: misc
# requires: endf
# slow: yes

## Introduction [intro]
In this example we use `ncmat2endf` to export NCrystal data into the [ENDF-6 format](https://www.nndc.bnl.gov/endfdocs/ENDF-102-2023.pdf).

- https://github.com/mctools/ncrystal/wiki/ncmat2endf

- J.I. Marquez Damian, T. Kittelmann, D. Di Julio, S. Xu, D. Roubtsov and G. Muhrer. Open source pipelines for thermal scattering data generation: from NCrystal to ENDF-6, 16th Conference of Nuclear Data for Science and Technology, Madrid, June 2025.

- G. Schnabel, D. L. Aldama, R. Capote, https://doi.org/10.48550/arXiv.2312.08249


## Example: export ENDF-6 file from NCMAT [export]

Ncmat2endf can convert most of the files in the NCrystal data library. Some of them cannot be converted either because they include physics that is not implemented in the ENDF-6 format, or require features that have not been yet implemented in ncmat2endf (e.g.: converting scattering kernels for gasses and liquids). The available data library can be listed with `NCrystal.browseFiles()` function, and it is available in the [NCrystal documentation](https://github.com/mctools/ncrystal/wiki/Data-library):

In [ ]:
NC.browseFiles(dump=True, factory='stdlib')

To export ENDF files we need to import the `ncmat2endf` module of NCrystal and call the `ncmat2endf()` function. This function takes as required input a [configuration string](https://github.com/mctools/ncrystal/wiki/CfgRefDoc), which includes the ncmat file to convert as well as parameters that modify the material (temperature, density, which components of the material we want to output). As an example, this converts `LiH_sg225_LithiumHydride.ncmat`:

In [ ]:
from NCrystal import ncmat2endf
res = ncmat2endf.ncmat2endf('LiH_sg225_LithiumHydride.ncmat;temp=293.6K',
                            material_name='LiH', force=True)

The result of the execution is a dictionary with information about the material, which is useful to use these libraries in a radiation transport simulation:

In [ ]:
import pprint
pprint.pprint(res)

In [ ]:
def dump_endf_files(res, nlines=100):
  for entry in res['files']:
      print(80*'-')
      print(entry['file'])
      with open(entry['file']) as f:
          lines = [next(f) for _ in range(nlines)]
      print("".join(lines))
      print('--- file continues ---'.center(80))
      print(80*'-','\n\n')
dump_endf_files(res)

The file includes generic metadata ("MyLab", "MyLib"). To customize this the `ncmat2endf` module includes the `EndfMetaData` class:

In [ ]:
m = ncmat2endf.EndfMetaData()
help(m)

As an example, we will set the laboratory name to IAEA, the author to Marie Curie, and assign material number 3001 to Li in LiH, and 3002 to H in LiH. We will also set the evaluation, revision and distribution dates to the current month:

In [ ]:
m.update_from_dict({'alab':'IAEA', 'auth': 'Marie Curie',
                    'matnum':{'Li':3001, 'H':3002},
                    'libname':'IAEA-TSL'})
m.set_all_dates_as_now()
res = ncmat2endf.ncmat2endf('LiH_sg225_LithiumHydride.ncmat;temp=293.6K',
                            endf_metadata=m,
                            material_name='LiH', force=True)
dump_endf_files(res, nlines=10)

## Command line interface [cli]

The same can be achieved using the command line interface. This is accessed with the `ncrystal_ncmat2endf` command:

In [ ]:
!ncrystal_ncmat2endf -h

ENDF metadata can be provided in the command line using a JSON dictionary or adding individual items:

In [ ]:
!ncrystal_ncmat2endf --mdata help

In [ ]:
!ncrystal_ncmat2endf 'LiH_sg225_LithiumHydride.ncmat;temp=293.6K' --name 'LiH' -f --now \
      -m alab:IAEA -m auth:"Marie Curie" -m matnum:"Li:3001,H:3002" -m libname:'IAEA-TSL' \
      --index results.json

The composition data that is returned by the `ncmat2endf()` function can be output in the command line using the `--index` option:

In [ ]:
!cat results.json